# 06 · From continuous values to states, events, and synchrony

Some questions are about episodes rather than raw values. Here a heat cube
becomes a standard state Dataset, contiguous states become event objects, and
occurrence synchrony measures where heat episodes co-occur with a reference
pixel. These are specialized verbs built on the same pipe protocol.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

# The two pulse groups create two heat episodes. Adding a spatial offset makes
# some pixels cross the threshold sooner or remain active longer than others.
time = pd.date_range("2025-07-01", periods=14, freq="D")
y = [40.2, 40.0, 39.8]
x = [-105.2, -105.0, -104.8]
pulse = np.array([0, 0, 5, 7, 6, 0, 0, 4, 6, 7, 5, 0, 0, 0])[:, None, None]
spatial = np.array([[-1.0, 0.0, 0.5], [-0.5, 1.0, 1.5], [-1.0, 0.5, 2.0]])[None, :, :]
cube = xr.DataArray(
    29 + pulse + spatial,
    dims=("time", "y", "x"),
    coords={"time": time, "y": y, "x": x},
    name="daily_max_temperature",
    attrs={"units": "degC"},
)

# threshold_state converts continuous values into a standard Dataset containing
# boolean state, exceedance magnitude, and the threshold used for classification.
states = (
    pipe(cube)
    | v.threshold_state(threshold=34.0, direction="above", name="hot_day")
).unwrap()

# detect_events labels contiguous runs. min_duration=2 removes isolated hot
# days; max_gap=0 means a cool day always separates two events.
events = (pipe(states) | v.detect_events(min_duration=2, max_gap=0)).unwrap()

# Compare every pixel's hot-day occurrence with the center pixel. The result is
# a map where 1 means identical occurrence and 0 means no shared occurrence.
synchrony = (
    pipe(states)
    | v.occurrence_synchrony(spatial_mode="reference", reference="center")
).unwrap()

# EventResult deliberately separates a cube-like Dataset from a tabular event
# catalog. Accumulate catalog rows here to make an event-count map for teaching.
event_count = xr.zeros_like(cube.isel(time=0), dtype=int)
for row in events.catalog.itertuples():
    event_count.values[row.y_index, row.x_index] += 1
sync_map = synchrony["occurrence_synchrony"].isel(time_window_end=0)

# These checks document the expected output contracts before visualization.
assert states["state"].dtype == bool
assert len(events.catalog) > 0

# The four panels tell the full progression: value → state → event → relation.
fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
cube.mean(("y", "x")).plot(ax=axes[0, 0], marker="o", color="#8b543c")
axes[0, 0].axhline(34, color="0.3", linestyle="--", label="threshold")
axes[0, 0].legend()
axes[0, 0].set_title("Continuous regional temperature")
states["state"].mean(("y", "x")).plot(ax=axes[0, 1], marker="o", color="#3f6f72")
axes[0, 1].set_title("v.threshold_state: active fraction")
event_count.plot(ax=axes[1, 0], cmap="YlOrRd", vmin=0)
axes[1, 0].set_title("v.detect_events: events per pixel")
sync_map.plot(ax=axes[1, 1], cmap="viridis", vmin=0, vmax=1)
axes[1, 1].set_title("v.occurrence_synchrony: reference map")
plt.show()